In [1]:
import sys
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.types import StructField, StructType, IntegerType, DoubleType, StringType, FloatType, DateType
from pyspark.sql.functions import to_date, col, lit, desc, sum, mean, round, dense_rank, row_number

In [2]:
# Configuración y creación de la Sesión de Spark
spark = SparkSession.builder.appName("Solucion_Tarea_1").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/25 22:05:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
schema_cyclist = StructType([
    StructField("cedula", IntegerType(), True),
    StructField("nombre", StringType(), True),
    StructField("provincia", StringType(), True)
])

schema_activity = StructType([
    StructField("cod_ruta", IntegerType(), True),
    StructField("cedula", IntegerType(), True),
    StructField("fecha", DateType(), True)
])

schema_route = StructType([
    StructField("cod_ruta", IntegerType(), True),
    StructField("nombre_ruta", StringType(), True),
    StructField("distancia", FloatType(), True)
])

schema_correct = StructType([
            StructField("cedula", IntegerType(), True),
            StructField("nombre", StringType(), True),
            StructField("provincia", StringType(), True),
            StructField("cod_ruta", IntegerType(), True),
            StructField("fecha", StringType(), True),
            StructField("nombre_ruta", StringType(), True),
            StructField("distancia", FloatType(), True)
])

schema_correct = StructType([
            StructField("cedula", IntegerType(), True),
            StructField("nombre", StringType(), True),
            StructField("provincia", StringType(), True),
            StructField("cod_ruta", IntegerType(), True),
            StructField("fecha", StringType(), True),
            StructField("nombre_ruta", StringType(), True),
            StructField("distancia", FloatType(), True)
])

df_cyclists = spark.read.csv("ciclistas.csv", schema=schema_cyclist, header=False, nullValue="NA")
df_activities = spark.read.csv("actividad.csv", schema=schema_activity, header=False, nullValue="NA")
df_routes = spark.read.csv("ruta.csv", schema=schema_route, header=False, nullValue="NA")

In [4]:
def join_cyclist_activity_route(df_cyclists, df_activities, df_routes):

    # PARTE 1: Actividad y Ruta
    # Debido a que el dataframe de "actividad" siempre debe tener un código de ruta, y no interesa una ruta que
    # no haya sido transitada en alguna actividad, el inner join es suficiente entre estos dos dataframes
    df_activities_routes = df_activities.join(df_routes, on="cod_ruta", how="inner")

    # PARTE 2: Ciclistas y Actividad/Ruta
    # Entre estos dos dataframes interesa preservar todos los ciclistas, hayan realizado alguna actividad o no.
    # Por otro lado, es posible que el mismo ciclista haya realizado la misma ruta más de una vez en el mismo día.
    # Por lo tanto, conviene primero ejecutar un right join entre ciclistas/actividad para preservar todas las actividades
    df_activities_cyclists = df_cyclists.join(df_activities_routes, on="cedula", how="right")

    # PARTE 3: Ciclistas nuevos
    # Se incluyen ahora los ciclistas que no han realizado ninguna actividad, esto por medio de un left anti join
    # entre el dataframe de ciclistas y el dataframe obtenido de la primera parte: Actividad/Ruta
    df_new_cyclists = df_cyclists.join(df_activities_routes, on="cedula", how="left_anti")

    # PARTE 4: Unión entre los dataframes de las partes 2 y 3
    # Se construye el dataframe completo al unir verticalmente los dataframes de todas las actividades con los datos
    # completos y los ciclistas nuevos. Primero, se debe agregar al dataframe de ciclistas nuevos, las columnas faltantes.
    df_new_cyclists = df_new_cyclists.withColumn("cod_ruta", lit(None).cast(IntegerType()))
    df_new_cyclists = df_new_cyclists.withColumn("fecha", lit(None).cast(DateType()))
    df_new_cyclists = df_new_cyclists.withColumn("nombre_ruta", lit(None).cast(StringType()))
    df_new_cyclists = df_new_cyclists.withColumn("distancia", lit(None).cast(FloatType()))
    # Se hace el join para obtener el dataframe completo
    df_complete = df_activities_cyclists.unionByName(df_new_cyclists)

    return df_complete

In [5]:
df_complete = join_cyclist_activity_route(df_cyclists, df_activities, df_routes)

In [6]:
df_complete.show()

+---------+-----------------+------------+--------+----------+--------------------+---------+
|   cedula|           nombre|   provincia|cod_ruta|     fecha|         nombre_ruta|distancia|
+---------+-----------------+------------+--------+----------+--------------------+---------+
|407490548| 'Esteban Bernal'|   'Heredia'| 2653871|2024-09-29|'Heredia - Monte ...|     52.4|
|403470532| 'Héctor Herrera'|   'Heredia'| 9638454|2024-07-01|'Ciudad Colón - E...|     98.6|
|403470532| 'Héctor Herrera'|   'Heredia'| 9856321|2024-08-12|'Belén - San Rafael'|     11.9|
|403470532| 'Héctor Herrera'|   'Heredia'| 1800689|2024-02-13|'San Pedro - Tres...|     46.9|
|403470532| 'Héctor Herrera'|   'Heredia'| 9836914|2024-03-14|'Santa Cruz - Lib...|     37.7|
|403470532| 'Héctor Herrera'|   'Heredia'| 9475635|2024-04-15|    'Tournón - Poás'|     90.4|
|403470532| 'Héctor Herrera'|   'Heredia'| 2653871|2024-07-16|'Heredia - Monte ...|     52.4|
|403470532| 'Héctor Herrera'|   'Heredia'| 9638745|2024-08-1

In [7]:
print(f"DataFrame Shape: ({df_complete.count()}, {len(df_complete.columns)})")

DataFrame Shape: (734, 7)


In [8]:
def aggregate_kms_per_cyclist_and_date(df_cyclist_activity_route):

    df_result = df_cyclist_activity_route.groupBy("cedula",
                                                  "nombre",
                                                  "provincia",
                                                  "fecha")\
                                         .agg(sum("distancia").alias("km"))
    df_result = df_result.withColumn("km", round(df_result.km, 2))

    return df_result

In [9]:
correct_1_df = spark.createDataFrame([(207580425, 'Alonso Arguedas', 'Heredia', 484, "2024-02-18", 'Belén - Poás', 45.3),
                                              (209190484, 'Elias Herrera', 'Alajuela', 703, '2024-03-21', 'San José de la Montaña - Sabanilla', 35.8),
                                              (408470591, 'Marco Jimenez', 'Heredia', 795, '2024-04-22', 'Escazú - Cartago', 28.3),
                                              (301780416, 'Javier Hernandez', 'Cartago', 932, '2024-05-08', 'Hatillo - Sabana', 10.6),
                                              (118740876, 'Gabriel Vargas', 'San José', 165, '2024-06-09', 'Paraíso - Alajuela', 60.4),
                                              (305840417, 'Fabián Solera', 'Cartago', 795, '2024-06-12', 'Escazú - Cartago', 28.3),
                                              (601980756, 'Josue Granados', 'Puntarenas', 484, '2024-07-01', 'Belén - Poás', 45.3),
                                              (705280357, 'Andrés Johnson', 'Limón', 165, '2024-08-09', 'Paraíso - Alajuela', 60.4),
                                              (601980756, 'Josue Granados', 'Puntarenas', 703, '2024-10-23', 'San José de la Montaña - Sabanilla', 35.8),
                                              (301780416, 'Javier Hernandez', 'Cartago', 484, '2024-10-05', 'Belén - Poás', 45.3),
                                              (408470591, 'Marco Jimenez', 'Heredia', 795, '2024-11-17', 'Escazú - Cartago', 28.3),
                                              (209190484, 'Elias Herrera', 'Alajuela', 484, '2024-06-13', 'Belén - Poás', 45.3),
                                              (209190484, 'Elias Herrera', 'Alajuela', 484, '2024-06-13', 'Belén - Poás', 45.3),
                                              (207580425, 'Alonso Arguedas', 'Heredia', 932, '2024-08-26', 'Hatillo - Sabana', 10.6),
                                              (209190484, 'Elias Herrera', 'Alajuela', 795, '2024-09-29', 'Escazú - Cartago', 28.3),
                                              (305840417, 'Fabián Solera', 'Cartago', 703, '2024-12-14', 'San José de la Montaña - Sabanilla', 35.8),
                                              (116570249, 'Rebeca Chaves', 'Heredia', None, None, None, None)],
                                              schema = schema_correct)

correct_1_df = correct_1_df.withColumn("fecha", to_date(col("fecha")))

correct_2_df = spark.createDataFrame([(408450338,'Luis Retana','Heredia', 9856712, '2026-12-02','Grecia - Santa Bárbara',34.1),
                                      (103680539,'Karen Pedregales','Heredia', 5986387, '2026-09-07','San Ramón - San Vicente',121.2),
                                      (109480531,'Leonora Avante','San José', 8574581, '2026-08-14','San Rafael - Naranjo',45.7),
                                              (509060541,'Luis Cruz','Guanacaste', 2895769, '2026-03-19','Liberia - San José',170.2),
                                              (302840639,'Gilberto Andrade','Cartago', 3108542, '2026-09-06','Heredia - Grecia',35.5),
                                              (307670615,'Wilmer Gutiérrez','Cartago', 5986387, '2026-10-21','San Ramón - San Vicente',121.2),
                                              (307670615,'Wilmer Gutiérrez','Cartago', 8574581, '2026-08-23','San Rafael - Naranjo',45.7),
                                              (109480531,'Leonora Avante','San José', 2895769, '2026-07-07','Liberia - San José',170.2),
                                              (302840639,'Gilberto Andrade','Cartago', 5986387, '2026-06-01','San Ramón - San Vicente',121.2),
                                              (509060541,'Luis Cruz','Guanacaste', 9856712, '2026-11-11','Grecia - Santa Bárbara',34.1),
                                              (307670615,'Wilmer Gutiérrez','Cartago', 3108542, '2026-06-15','Heredia - Grecia',35.5),
                                              (109480531,'Leonora Avante','San José', 5986387, '2026-05-24','San Ramón - San Vicente',121.2)],
                                              schema = schema_correct)

correct_2_df = correct_2_df.withColumn("fecha", to_date(col("fecha")))

In [11]:
df_agg_result_t1 = aggregate_kms_per_cyclist_and_date(correct_1_df)
df_agg_result_t1.show()

+---------+----------------+----------+----------+----+
|   cedula|          nombre| provincia|     fecha|  km|
+---------+----------------+----------+----------+----+
|207580425| Alonso Arguedas|   Heredia|2024-02-18|45.3|
|209190484|   Elias Herrera|  Alajuela|2024-03-21|35.8|
|408470591|   Marco Jimenez|   Heredia|2024-04-22|28.3|
|301780416|Javier Hernandez|   Cartago|2024-05-08|10.6|
|118740876|  Gabriel Vargas|  San José|2024-06-09|60.4|
|305840417|   Fabián Solera|   Cartago|2024-06-12|28.3|
|601980756|  Josue Granados|Puntarenas|2024-07-01|45.3|
|705280357|  Andrés Johnson|     Limón|2024-08-09|60.4|
|601980756|  Josue Granados|Puntarenas|2024-10-23|35.8|
|301780416|Javier Hernandez|   Cartago|2024-10-05|45.3|
|408470591|   Marco Jimenez|   Heredia|2024-11-17|28.3|
|209190484|   Elias Herrera|  Alajuela|2024-06-13|90.6|
|207580425| Alonso Arguedas|   Heredia|2024-08-26|10.6|
|209190484|   Elias Herrera|  Alajuela|2024-09-29|28.3|
|305840417|   Fabián Solera|   Cartago|2024-12-1

In [61]:
df_agg_result = aggregate_kms_per_cyclist_and_date(df_complete)

In [63]:
print(f"DataFrame Shape: ({df_agg_result.count()}, {len(df_agg_result.columns)})")

DataFrame Shape: (723, 5)


In [206]:
def top_n_cyclists(df_partial_aggregate, n):

    df_partial_aggregate_clean = df_partial_aggregate.dropna()

    df_result = df_partial_aggregate_clean.groupBy("provincia", "cedula", "nombre")\
                                          .agg(sum("km").alias("km_total"),
                                               mean("km").alias("km_prom_diario"))

    df_result = df_result.withColumn("km_total", round(df_result.km_total, 2))\
                         .withColumn("km_prom_diario", round(df_result.km_prom_diario, 2))

    window_spec = Window.partitionBy("provincia")\
                        .orderBy(col("km_total").desc(), col("km_prom_diario").desc())
    
    df_result = df_result.withColumn("rank", dense_rank().over(window_spec))\
                         .filter(col("rank") <= n)\
                         .select("rank", "provincia", "cedula", "nombre", "km_total", "km_prom_diario")

    return df_result

In [210]:
top_n = top_n_cyclists(df_agg_result_t1, 3)
top_n.show()

+----+----------+---------+----------------+--------+--------------+
|rank| provincia|   cedula|          nombre|km_total|km_prom_diario|
+----+----------+---------+----------------+--------+--------------+
|   1|  Alajuela|209190484|   Elias Herrera|   154.7|         51.57|
|   1|   Cartago|305840417|   Fabián Solera|    64.1|         32.05|
|   2|   Cartago|301780416|Javier Hernandez|    55.9|         27.95|
|   1|   Heredia|408470591|   Marco Jimenez|    56.6|          28.3|
|   2|   Heredia|207580425| Alonso Arguedas|    55.9|         27.95|
|   1|     Limón|705280357|  Andrés Johnson|    60.4|          60.4|
|   1|Puntarenas|601980756|  Josue Granados|    81.1|         40.55|
|   1|  San José|118740876|  Gabriel Vargas|    60.4|          60.4|
+----+----------+---------+----------------+--------+--------------+

